In [1]:
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

In [2]:
DATA_PATH = "../data/movie_dataset_clean.parquet"

df = pd.read_parquet(DATA_PATH)

df.head()

,id,title,overview,genres,cast,directors,keywords,release_date,vote_average,vote_count,runtime,year,document
0,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","Animation, Family","Albert Brooks, Ellen DeGeneres, Alexander Goul...","Andrew Stanton, Lee Unkrich","sydney, australia, parent child relationship, ...",2003-05-30 00:00:00,7.824,18061,100,2003.0,"Title: Finding Nemo\nOverview: Nemo, an advent..."
1,14,American Beauty,"Lester Burnham, a depressed suburban father in...",Drama,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"estate agent, adultery, coming out, first time...",1999-09-15 00:00:00,8.0,11260,122,1999.0,Title: American Beauty\nOverview: Lester Burnh...
2,16,Dancer in the Dark,"Selma, a Czech immigrant on the verge of blind...","Drama, Crime","Björk, Catherine Deneuve, David Morse, Peter S...",Lars von Trier,"factory worker, dying and death, individual, i...",2000-06-30 00:00:00,7.869,1618,141,2000.0,"Title: Dancer in the Dark\nOverview: Selma, a ..."
3,17,The Dark,"In an attempt to pull her family together, Adè...","Horror, Thriller, Mystery","Maria Bello, Sean Bean, Abigail Stone, Richard...",John Fawcett,"sea, wales, child abuse, shepherd, adolescence...",2005-09-28 00:00:00,5.755,247,87,2005.0,Title: The Dark\nOverview: In an attempt to pu...
4,20,My Life Without Me,A fatally ill mother with only two months to l...,"Drama, Romance","Sarah Polley, Amanda Plummer, Scott Speedman, ...",Isabel Coixet,"dying and death, daughter, farewell, night shi...",2003-03-07 00:00:00,5.941,421,106,2003.0,Title: My Life Without Me\nOverview: A fatally...


In [3]:
df.shape

(12338, 13)

In [4]:
EMBEDDING_PATH = "../data/movie_embeddings.npy"

embeddings = np.load(
    EMBEDDING_PATH
)

In [5]:
embeddings.shape

(12338, 384)

In [6]:
assert len(df) == len(embeddings)

print("Dataset and embeddings match!")

Dataset and embeddings match!


In [7]:
client = QdrantClient(
    path="../qdrant_db"
)

In [8]:
#create collection
COLLECTION_NAME = "movies"

In [9]:
client.collection_exists(
    COLLECTION_NAME
)

False

In [10]:
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embeddings.shape[1],
        distance=Distance.COSINE
    )
)

True

In [13]:
#create points
points = []

for idx, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    point = PointStruct(

        id=int(idx),

        vector=embeddings[idx].tolist(),

        payload={
            "title": row["title"],
            "overview": row["overview"],
            "genres": row["genres"],
            "directors": row["directors"],
            "cast": row["cast"],
            "keywords": row["keywords"],
            "release_year": row["release_date"],
            "rating": float(row["vote_average"]),
            "runtime": int(row["runtime"])
        }
    )

    points.append(point)

  0%|          | 0/12338 [00:00<?, ?it/s]

In [14]:
len(points)

12338

In [16]:
#upload to qdrant
BATCH_SIZE = 100

In [17]:
for i in tqdm(
    range(0, len(points), BATCH_SIZE)
):

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i+BATCH_SIZE]
    )

  0%|          | 0/124 [00:00<?, ?it/s]

In [18]:
client.get_collection(
    COLLECTION_NAME
)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=12338, segments_count=1, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimiz

In [20]:
#test retreival
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)
query = "movies about space exploration and astronauts"
query_vector = model.encode(
    query,
    normalize_embeddings=True
)

/Users/phoothwincho/anaconda3/envs/movie-rag-new/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
#search
results = client.query_points(

    collection_name=COLLECTION_NAME,

    query=query_vector.tolist(),

    limit=5
)

In [22]:
#View results
for result in results.points:

    print(
        result.payload["title"],
        "|",
        result.payload["rating"]
    )

Return to Space | 6.514
Gravity | 7.162
3022 | 5.402
Interstellar | 8.417
Red Planet | 5.691


In [23]:
#test other query

query = "mind bending movie about dreams"

query_vector = model.encode(
    query,
    normalize_embeddings=True
)


results = client.query_points(
    collection_name="movies",
    query=query_vector.tolist(),
    limit=5
)


for result in results.points:
    print(
        result.payload["title"]
    )

The Science of Sleep
On Body and Soul
Inception
In Dreams
The Diabolical
